In [ ]:
import logfire
from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool

from medici.agents.agentic.grader import GraderAgent
from medici.agents.agentic.planner import PlannerAgent
from medici.agents.agentic.query_expander import QueryExpander
from medici.agents.agentic.query_rewriter import QueryRewriter
from medici.agents.agentic.router import RouterAgent
from medici.agents.agentic.synthesizer import SynthesizerAgent
from medici.agents.graph.graph import compile_graph_with_postgres
from medici.agents.graph.runner import GraphPipeline
from medici.agents.memory.short_term import ShortTermMemoryManager
from medici.agents.retrieval import RetrievalAgent
from medici.common.llm.gemini import GeminiClient
from medici.common.llm.groq import GroqClient
from medici.common.services.hybrid_search import HybridSearch
from medici.common.services.qdrant import QdrantStorageService
from medici.common.services.reranker import Reranker
from medici.common.utils.config import config
from medici.ingestion.embedding import EmbeddingService

In [ ]:
logfire.configure(service_name="Quering")

In [ ]:
pool = AsyncConnectionPool(
    conninfo=config.POSTGRES_CONN_STRING,
    min_size=2,
    max_size=10,
    open=False,
    kwargs={"autocommit": True, "row_factory": dict_row},
)

In [ ]:
await pool.open()

In [ ]:
gemini_client = GeminiClient(timeout_seconds=30, max_retries=2, model="gemini-2.5-flash")
groq_client = GroqClient(timeout_seconds=30, max_retries=2)

In [ ]:
short_term = ShortTermMemoryManager(config.REDIS_URL)

In [ ]:
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
)

In [ ]:
storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=embedding_service.vector_size,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [ ]:
hybrid_search = HybridSearch(storage_service=storage_service, embedding_service=embedding_service)
reranker = Reranker()
query_expander = QueryExpander(gemini_client)

In [ ]:
retrieval_agent = RetrievalAgent(
    llm_client=groq_client,
    hybrid_search=hybrid_search,
    reranker=reranker,
    query_expand=query_expander,
)

In [ ]:
graph = await compile_graph_with_postgres(
    pool=pool,
    short_term=short_term,
    rewriter=QueryRewriter(gemini_client),
    router=RouterAgent(groq_client),
    planner=PlannerAgent(gemini_client),
    retrieval_agent=retrieval_agent,
    grader=GraderAgent(groq_client),
    synthesizer=SynthesizerAgent(groq_client),
)

In [ ]:
graph

In [ ]:
pipeline = GraphPipeline(graph, short_term_memory=short_term)

In [ ]:
chat1 = await pipeline.chat(
    user_message="What is transformer in LLM?",
    session_id="8297137d-4e46-4337-8fa7-3b99fe92d298",
    user_id="14684",
)

In [ ]:
chat1